# Lesson 3.6 — How Do I Handle Tables, Code Blocks, and Lists?

**Companion notebook for the blog post.**

Naive chunking splits text at fixed character counts — it doesn't know or care that it's slicing through the middle of a table, cutting a code block in half, or separating a heading from the paragraph it belongs to.

This notebook builds a **structure-aware Markdown parser** that understands heading hierarchy, code fences, and tables — and shows you how to turn that structure into better chunks for RAG.

### What you'll learn:
- Why splitting Markdown naively destroys its structure
- How to parse Markdown preserving headings, code blocks, and tables
- What a **breadcrumb** is and why it helps retrieval
- How structure-aware chunks compare to fixed-size chunks in practice

---
## 📦 Setup

In [ ]:
import os
os.environ["USE_TF"] = "0"

import re
import json
from pprint import pprint

---
## 🗂️ Sample Document

We need a realistic Markdown document to work with — one that has multiple heading levels, code blocks, tables, and prose. The document below describes a fictional RAG system architecture, which conveniently covers all the tricky cases.

In [ ]:
SAMPLE_MARKDOWN = """
# RAG System Architecture Guide

This guide covers the design and implementation of a production-grade
Retrieval-Augmented Generation system.

## 1. Ingestion Pipeline

The ingestion pipeline converts raw documents into searchable vector embeddings.
It runs offline and populates the vector database.

### 1.1 Document Loading

Use the appropriate loader for each document type.
Mixing loaders for the wrong format silently degrades quality.

| Format | Recommended Loader | Notes |
|--------|-------------------|-------|
| PDF    | PyMuPDF (fitz)    | Fast, handles most layouts |
| HTML   | BeautifulSoup     | Strip nav/footer before chunking |
| DOCX   | python-docx       | Preserves heading styles |
| CSV    | pandas            | Convert rows to prose before embedding |

### 1.2 Chunking

Split documents into chunks before embedding. Chunk size affects both
retrieval precision and the amount of context the LLM receives.

#### Chunking Parameters

Start with these defaults and tune based on your eval metrics:

| Parameter     | Default | Effect |
|---------------|---------|--------|
| chunk_size    | 512     | Larger = more context, noisier match |
| chunk_overlap | 64      | Prevents boundary information loss |
| separators    | [\\n\\n, \\n, '. '] | Controls split priority |

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=[\"\\n\\n\", \"\\n\", \". \", \" \", \"\"]
)
chunks = splitter.split_documents(docs)
```

## 2. Embedding and Storage

Embeddings are dense vector representations that capture semantic meaning.
Every chunk gets embedded and stored in a vector database.

### 2.1 Choosing an Embedding Model

Model choice depends on your latency, cost, and accuracy requirements.

| Model | Dims | Speed | Best for |
|-------|------|-------|----------|
| all-MiniLM-L6-v2 | 384 | Very fast | Prototypes, low-resource |
| text-embedding-3-small | 1536 | Fast | Production (OpenAI) |
| text-embedding-3-large | 3072 | Moderate | High-accuracy needs |

```python
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=\"all-MiniLM-L6-v2\",
    model_kwargs={\"device\": \"cpu\"}
)
```

### 2.2 Vector Database Options

Choose your vector database based on scale and operational needs.
For most RAG prototypes, ChromaDB running locally is sufficient.

## 3. Retrieval

At query time, the retriever finds the top-k most semantically similar chunks.

### 3.1 Similarity Search

The default retrieval strategy uses cosine similarity between the query embedding
and stored chunk embeddings.

```python
retriever = vectorstore.as_retriever(search_kwargs={\"k\": 3})
results = retriever.invoke(\"What chunk size should I use?\")
```

### 3.2 Hybrid Search

Combine semantic search with BM25 keyword search for better recall on
exact terms (product names, error codes, version numbers).

#### When to Use Hybrid Search

Use hybrid search when your documents contain:
exact identifiers like model names or error codes,
technical jargon that embeddings may not represent well,
or short queries where semantic similarity alone is unreliable.
"""

print(f"Document length: {len(SAMPLE_MARKDOWN)} characters")
print(f"Line count:      {len(SAMPLE_MARKDOWN.splitlines())} lines")

---
## 🔪 The Problem with Naive Chunking on Markdown

Before looking at the solution, let's see what goes wrong with a standard `RecursiveCharacterTextSplitter`.

In [ ]:
# Simulate naive fixed-size chunking
def naive_chunk(text, chunk_size=300, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


naive_chunks = naive_chunk(SAMPLE_MARKDOWN)

print(f"Naive chunking produced {len(naive_chunks)} chunks\n")

# Show two chunks that illustrate the problem
print("=" * 65)
print("Chunk 3 — a table split mid-row:")
print("=" * 65)
print(repr(naive_chunks[2]))

print()
print("=" * 65)
print("Chunk 5 — a code block split in the middle:")
print("=" * 65)
print(repr(naive_chunks[4]))

print()
print("⚠️  Neither chunk knows WHICH section it came from.")
print("    An LLM receiving chunk 3 has no idea this is about 'Chunking Parameters'.")

---
## 🏗️ Structure-Aware Markdown Parser

The parser below walks through Markdown line by line and tracks three things simultaneously:

1. **Heading hierarchy** — which `#`, `##`, `###`, `####` we're currently inside
2. **Code fences** — ```` ``` ```` open/close pairs, preserving language and content
3. **Table rows** — lines starting with `|`

Every time a new heading is encountered, the accumulated content is saved as a chunk — tagged with all the headings it sits under (the **breadcrumb**).

```
# RAG System Architecture Guide          ← H1
  ## 1. Ingestion Pipeline               ← H2  
    ### 1.2 Chunking                     ← H3
      #### Chunking Parameters           ← H4
        [table content here]
        [code block here]

Breadcrumb: "RAG System Architecture Guide > 1. Ingestion Pipeline > 1.2 Chunking > Chunking Parameters"
```

The breadcrumb is the key innovation — it gives every chunk full location context, even when read in isolation.

In [ ]:
import re

def parse_markdown_with_structure(md_text):
    """Parse markdown preserving heading hierarchy, code blocks, and tables."""

    chunks = []
    current_headings = {1: "", 2: "", 3: "", 4: ""}
    current_content = []
    in_code_block = False
    code_block_content = []
    code_language = ""

    for line in md_text.split("\n"):

        # ── Code fence detection ─────────────────────────────────────────────
        if line.strip().startswith("```"):
            if not in_code_block:
                # Opening fence — record the language tag (e.g. "python", "bash")
                in_code_block = True
                code_language = line.strip().replace("```", "").strip()
                code_block_content = []
            else:
                # Closing fence — save the complete code block as one item
                in_code_block = False
                code_text = "\n".join(code_block_content)
                current_content.append({
                    "type": "code",
                    "language": code_language,
                    "content": code_text
                })
            continue

        if in_code_block:
            code_block_content.append(line)
            continue

        # ── Heading detection ────────────────────────────────────────────────
        heading_match = re.match(r'^(#{1,4})\s+(.+)$', line)
        if heading_match:
            # Save everything accumulated so far as a chunk
            if current_content:
                chunks.append({
                    "content": current_content.copy(),
                    "headings": {k: v for k, v in current_headings.items() if v},
                    "breadcrumb": " > ".join(
                        v for k, v in sorted(current_headings.items()) if v
                    )
                })
                current_content = []

            level = len(heading_match.group(1))
            title = heading_match.group(2).strip()
            current_headings[level] = title
            # Clear all deeper heading levels — we've entered a new section
            for l in range(level + 1, 5):
                current_headings[l] = ""
            continue

        # ── Table row detection ──────────────────────────────────────────────
        if line.strip().startswith("|") and "|" in line:
            current_content.append({
                "type": "table_row",
                "content": line.strip()
            })
            continue

        # ── Regular text ─────────────────────────────────────────────────────
        if line.strip():
            current_content.append({
                "type": "text",
                "content": line.strip()
            })

    # Flush the last section
    if current_content:
        chunks.append({
            "content": current_content.copy(),
            "headings": {k: v for k, v in current_headings.items() if v},
            "breadcrumb": " > ".join(
                v for k, v in sorted(current_headings.items()) if v
            )
        })

    return chunks


chunks = parse_markdown_with_structure(SAMPLE_MARKDOWN)
print(f"✅ Parsed {len(chunks)} structured chunks")

---
## 🔍 Exploring the Chunks

Let's look at what the parser produced — chunk by chunk.

In [ ]:
# Overview: breadcrumb + content type summary for every chunk
print(f"{'#':<4} {'Breadcrumb':<65} {'Content types'}")
print("-" * 100)

for i, chunk in enumerate(chunks):
    type_counts = {}
    for item in chunk["content"]:
        t = item["type"]
        type_counts[t] = type_counts.get(t, 0) + 1
    summary = ", ".join(f"{v}x {k}" for k, v in type_counts.items())
    breadcrumb = chunk["breadcrumb"][:63]
    print(f"{i:<4} {breadcrumb:<65} {summary}")

In [ ]:
# Deep-dive: inspect any chunk by index
CHUNK_INDEX = 3   # 🎛️ change this to explore different chunks

chunk = chunks[CHUNK_INDEX]
print(f"Chunk {CHUNK_INDEX}")
print(f"Breadcrumb : {chunk['breadcrumb']}")
print(f"Headings   : {chunk['headings']}")
print(f"Items      : {len(chunk['content'])}")
print()

for item in chunk["content"]:
    t = item["type"]
    if t == "text":
        print(f"  [text]       {item['content'][:100]}")
    elif t == "code":
        lines = item['content'].splitlines()
        print(f"  [code/{item['language']:6s}] {lines[0][:80]}")
        if len(lines) > 1:
            print(f"               ... ({len(lines)} lines total)")
    elif t == "table_row":
        print(f"  [table_row]  {item['content'][:90]}")

---
## 🍞 The Breadcrumb — Why It Matters

The breadcrumb is what makes structure-aware chunking powerful for RAG.

Without it, a chunk about *chunking parameters* looks like this to the LLM:
```
| chunk_size | 512 | Larger = more context, noisier match |
| chunk_overlap | 64 | Prevents boundary information loss |
```

With the breadcrumb prepended, the same chunk becomes:
```
RAG System Architecture Guide > 1. Ingestion Pipeline > 1.2 Chunking > Chunking Parameters

| chunk_size | 512 | Larger = more context, noisier match |
| chunk_overlap | 64 | Prevents boundary information loss |
```

Now the LLM knows exactly where this table lives in the document — and can answer questions like *"what chunk_size does the ingestion pipeline use?"* correctly.

In [ ]:
# Show all unique breadcrumbs — the document's skeleton
print("Document skeleton (all breadcrumbs):")
print("=" * 65)
for i, chunk in enumerate(chunks):
    bc = chunk["breadcrumb"] or "(top-level)"
    # Visual indentation based on depth
    depth = bc.count(" > ")
    indent = "  " * depth
    leaf = bc.split(" > ")[-1] if bc != "(top-level)" else bc
    print(f"  {indent}[{i}] {leaf}")

---
## 🔄 Converting to LangChain Documents

To use these chunks in a RAG pipeline, we need to convert them to LangChain `Document` objects. The key decisions:

- **`page_content`**: the text that gets **embedded** — breadcrumb + rendered content
- **`metadata`**: the structure that gets **stored** alongside the vector (headings, content types, source)

In [ ]:
from langchain.schema import Document


def render_chunk_content(items):
    """Convert structured content items back into a flat, readable string."""
    parts = []
    table_rows = []

    for item in items:
        if item["type"] == "text":
            # Flush any accumulated table rows first
            if table_rows:
                parts.append("\n".join(table_rows))
                table_rows = []
            parts.append(item["content"])

        elif item["type"] == "table_row":
            table_rows.append(item["content"])

        elif item["type"] == "code":
            if table_rows:
                parts.append("\n".join(table_rows))
                table_rows = []
            lang = item.get("language", "")
            parts.append(f"```{lang}\n{item['content']}\n```")

    if table_rows:
        parts.append("\n".join(table_rows))

    return "\n\n".join(parts)


def chunks_to_documents(chunks, source="sample_doc.md"):
    """Convert parsed chunks into LangChain Document objects."""
    documents = []

    for i, chunk in enumerate(chunks):
        rendered = render_chunk_content(chunk["content"])
        content_types = list({item["type"] for item in chunk["content"]})

        # Prepend breadcrumb so the embedding captures location context
        if chunk["breadcrumb"]:
            page_content = f"{chunk['breadcrumb']}\n\n{rendered}"
        else:
            page_content = rendered

        documents.append(Document(
            page_content=page_content,
            metadata={
                "source":        source,
                "chunk_index":   i,
                "breadcrumb":    chunk["breadcrumb"],
                "headings":      chunk["headings"],
                "content_types": content_types,
                "has_code":      "code"       in content_types,
                "has_table":     "table_row"  in content_types,
            }
        ))

    return documents


documents = chunks_to_documents(chunks)

print(f"✅ Converted {len(documents)} chunks to LangChain Documents\n")

# Preview one document
print("=" * 65)
print("Example document (chunk 3):")
print("=" * 65)
doc = documents[3]
print(f"page_content:\n{doc.page_content}")
print(f"\nmetadata:")
for k, v in doc.metadata.items():
    print(f"  {k}: {v}")

---
## ⚖️ Naive vs Structure-Aware: Side-by-Side

The same question answered with two different sets of chunks.

In [ ]:
QUESTION = "What chunk size and overlap should I use?"

# ── Naive chunks ────────────────────────────────────────────────────────────
# Find the naive chunk most likely to be retrieved for this question
# (we simulate by searching for keywords)
keyword_hits = [
    (i, c) for i, c in enumerate(naive_chunks)
    if "chunk_size" in c or "chunk_overlap" in c
]

print(f'Question: "{QUESTION}"')
print()
print("❌ Best naive chunk retrieved:")
print("-" * 65)
if keyword_hits:
    idx, best = keyword_hits[0]
    print(f"Chunk #{idx}")
    print(best)
    print("\n⚠️  The LLM gets the table rows — but has NO idea this is about")
    print("    the Ingestion Pipeline's Chunking section.")

print()
print("=" * 65)

# ── Structure-aware chunks ───────────────────────────────────────────────────
struct_hits = [
    doc for doc in documents
    if "chunk_size" in doc.page_content or "chunk_overlap" in doc.page_content
]

print()
print("✅ Best structure-aware chunk retrieved:")
print("-" * 65)
if struct_hits:
    doc = struct_hits[0]
    print(f"Breadcrumb: {doc.metadata['breadcrumb']}")
    print(f"Has code:   {doc.metadata['has_code']}")
    print(f"Has table:  {doc.metadata['has_table']}")
    print(f"\npage_content:\n{doc.page_content}")
    print("\n✅ The LLM gets the full context: breadcrumb + table + code example.")

---
## 🔎 Metadata Filtering in Action

Structured metadata unlocks filtering that naive chunking can never offer.
You can ask the retriever to only search code-containing chunks, or only table-containing chunks.

In [ ]:
# Filter by content type — useful in a vector DB that supports metadata filters
code_docs  = [d for d in documents if d.metadata["has_code"]]
table_docs = [d for d in documents if d.metadata["has_table"]]
text_only  = [d for d in documents if not d.metadata["has_code"] and not d.metadata["has_table"]]

print(f"Total documents : {len(documents)}")
print(f"With code blocks: {len(code_docs)}")
print(f"With tables     : {len(table_docs)}")
print(f"Text only       : {len(text_only)}")

print("\n--- Code-containing chunks ---")
for d in code_docs:
    print(f"  [{d.metadata['chunk_index']}] {d.metadata['breadcrumb']}")

print("\n--- Table-containing chunks ---")
for d in table_docs:
    print(f"  [{d.metadata['chunk_index']}] {d.metadata['breadcrumb']}")

---
## 🗃️ Heading Hierarchy Filter

You can also filter by heading level — for example, only retrieve chunks from a specific top-level section.

In [ ]:
TARGET_SECTION = "2. Embedding and Storage"  # 🎛️ change to filter by section

section_docs = [
    d for d in documents
    if TARGET_SECTION in d.metadata["breadcrumb"]
]

print(f'Chunks inside "{TARGET_SECTION}": {len(section_docs)}\n')
for d in section_docs:
    print(f"  [{d.metadata['chunk_index']}] {d.metadata['breadcrumb']}")
    print(f"       content_types: {d.metadata['content_types']}")
    print()

---
## 🧪 Try It on Your Own Markdown

Paste any Markdown document and inspect how it gets parsed.

In [ ]:
YOUR_MARKDOWN = """
# Your Document Title Here

Replace this with your own Markdown content.

## Section One

Some text here.

### Subsection

| Column A | Column B |
|----------|----------|
| value 1  | value 2  |

```python
print("hello world")
```
"""

your_chunks = parse_markdown_with_structure(YOUR_MARKDOWN)
your_docs   = chunks_to_documents(your_chunks, source="your_doc.md")

print(f"Parsed {len(your_chunks)} chunks:\n")
for i, d in enumerate(your_docs):
    print(f"  Chunk {i}: {d.metadata['breadcrumb'] or '(top-level)'}")
    print(f"    types: {d.metadata['content_types']}")
    print(f"    preview: {d.page_content[:80].replace(chr(10), ' ')}")
    print()

---
## 🧪 Exercises

1. **Add list support**: The parser currently ignores `- item` and `1. item` lines. Add a `"list_item"` type and handle them in `render_chunk_content`.
2. **Add H5/H6**: Extend `current_headings` to cover `#####` and `######`.
3. **Chunk on H2 only**: Modify the parser to only split on H2 headings, keeping H3/H4 content bundled with their parent section. When is this better?
4. **Real document**: Convert a Markdown file from your own project and inspect the breadcrumb tree. Are there any chunks that surprise you?